# Phase 2.2 Experiment 6: Does half the training data work better?

## Research question

Compare the Phase 2.2 CNN trained on the full training split with the same CNN trained on a random 50% subset of that training split. Both models use the same validation tracks and the same held-out test tracks.

## Controlled design

First, create one 81% train, 9% validation, and 10% test split from the full eligible GTZAN dataset. Then randomly select about half of the full training tracks, balancing the selection within each genre and keeping exact-content hash groups together. The full model trains on all training tracks; the smaller model trains only on the selected half. Validation and test tracks are unchanged and shared by both models.

The architecture, log-mel transforms, loss, optimizer, scheduler, early stopping, checkpoint rule, and seed match the Phase 2.2 baseline workflow. Validation metrics select each checkpoint; the shared test set is used only for final evaluation. The test predictions are paired track by track.

The smaller model sees about 40.5% of the full dataset for training, while both models retain the same 9% validation and 10% test sets. Exact duplicate groups can make the realized percentages differ slightly. The notebook reports the actual counts.

Run the cells from top to bottom. The first run downloads the GTZAN archive (about 1.23 GB) if audio is not already present and caches reusable features under artifacts.

## 1. Download and prepare GTZAN

This data setup follows the Phase 2.2 baseline: it downloads the Marsyas GTZAN archive when needed, extracts the ten genre folders, and skips the known corrupted jazz.00054.wav file.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path, PurePosixPath

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
from huggingface_hub import hf_hub_download

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content")
else:
    current_and_parent_dirs = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next(
        (path for path in current_and_parent_dirs if (path / "filtered_splits").is_dir()),
        Path.cwd(),
    )
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
GENRE_FOLDERS = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]

# returns a dictionary of each genre and how many files are stored under its name.
def genre_file_counts(root):
    return {
        genre: len(list((root / genre).glob("*.wav")))
        for genre in GENRE_FOLDERS
    }

# Creates the directory
# parents=True means create the directory if it doesn't exist
# exist_ok=True means reuse the directory if it exists already (don't raise an error)
DATA_DIR.mkdir(parents=True, exist_ok=True)
existing_counts = genre_file_counts(DATA_DIR)

# checks whether all values (counts) inside the genre counts dict are bigger or equal to 90.
# If so, skips the download
if all(count >= 90 for count in existing_counts.values()):
    print(f"GTZAN is already present at {DATA_DIR.resolve()} — skipping download.")
else:
    archive_path = hf_hub_download(
        repo_id="marsyas/gtzan",
        repo_type="dataset",
        filename="data/genres.tar.gz",
        local_dir=str(PROJECT_DIR / "data" / "downloads"),
    )
    print(f"Downloaded archive: {archive_path}")

    extracted_files = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in archive.getmembers():
            if not member.isfile():
                continue
            member_path = PurePosixPath(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in GTZAN archive: {member.name}")
            filename = member_path.name
            if not filename.lower().endswith(".wav") or filename.startswith("._"):
                continue
            genres_in_path = [
                part.lower() for part in member_path.parts[:-1]
                if part.lower() in GENRE_FOLDERS
            ]
            if not genres_in_path:
                continue
            genre = genres_in_path[-1]
            if genre == "jazz" and filename == "jazz.00054.wav":
                continue
            destination = DATA_DIR / genre / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            extracted_files += 1

    final_counts = genre_file_counts(DATA_DIR)
    missing_or_incomplete = {genre: count for genre, count in final_counts.items() if count < 90}
    if missing_or_incomplete:
        raise RuntimeError(
            f"GTZAN extraction looks incomplete: {missing_or_incomplete}. "
            "Check the Colab internet connection and rerun this cell."
        )
    print(f"Extracted {extracted_files} usable WAV files into {DATA_DIR.resolve()}")
    print("Files per genre:", final_counts)

print(f"Dataset directory ready: {DATA_DIR.resolve()}")

## 2. Imports, settings, and reproducibility

In [ ]:
import hashlib
import math
import random
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic kernels can make repeated runs easier to compare.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Audio and feature settings
PROJECT_DIR = PROJECT_DIR  # detected in the dataset setup cell above
DATA_DIR = PROJECT_DIR / "data" / "genres_original"
SAMPLE_RATE = 22_050
TRACK_SECONDS = 30
SEGMENT_SECONDS = 5
N_MELS = 128
N_FFT = 2_048
HOP_LENGTH = 512
DB_RANGE = 80.0

# Training settings
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 8
LEARNING_RATE = 1e-3

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "phase_2_2_experiment_6"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dataset folder: {DATA_DIR.resolve()}")

## 3. Index the audio and identify exact copies

In [ ]:
# A few common extraction layouts are checked before asking you to edit DATA_DIR.
if not DATA_DIR.exists():
    alternatives = [
        Path("genres_original"),
        Path("Data/genres_original"),
        Path("data/Data/genres_original"),
    ]
    DATA_DIR = next((path for path in alternatives if path.exists()), DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the GTZAN genre folders at {DATA_DIR.resolve()}. "
        "Extract the dataset and set DATA_DIR to the folder that directly contains "
        "the ten genre folders."
    )

audio_extensions = {".wav", ".au", ".mp3", ".flac", ".ogg"}
genre_dirs = sorted(path for path in DATA_DIR.iterdir() if path.is_dir())
CLASS_NAMES = [path.name for path in genre_dirs]
if len(CLASS_NAMES) != 10:
    raise ValueError(
        f"Expected 10 genre folders, found {len(CLASS_NAMES)}: {CLASS_NAMES}. "
        "Check that DATA_DIR points to the folder containing the genre folders."
    )

class_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as audio_file:
        for block in iter(lambda: audio_file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

records = []
for genre_dir in genre_dirs:
    for audio_path in sorted(genre_dir.iterdir()):
        if audio_path.is_file() and audio_path.suffix.lower() in audio_extensions:
            try:
                audio_info = sf.info(str(audio_path))
                file_hash = sha256_file(audio_path)
            except (OSError, RuntimeError, ValueError) as error:
                print(f"Skipping unreadable audio file {audio_path}: {error}")
                continue
            records.append(
                {
                    "path": audio_path.resolve(),
                    "track_name": audio_path.name,
                    "relative_path": PurePosixPath(genre_dir.name, audio_path.name).as_posix(),
                    "genre": genre_dir.name,
                    "label_id": class_to_id[genre_dir.name],
                    "duration_seconds": audio_info.duration,
                    "sha256": file_hash,
                }
            )

tracks = pd.DataFrame(records)
if tracks.empty:
    raise ValueError(
        f"No supported audio files found directly inside the genre folders in {DATA_DIR}."
    )

print(f"Found {len(tracks)} audio files across {tracks['genre'].nunique()} genres.")
display(tracks.head())

## 4. Inspect class balance, durations, and conflicting exact copies

In [ ]:
print("Files per genre:")
display(tracks.groupby("genre").size().rename("file_count").to_frame())

print(
    "\nDuration summary (seconds): "
    f"min={tracks['duration_seconds'].min():.2f}, "
    f"median={tracks['duration_seconds'].median():.2f}, "
    f"max={tracks['duration_seconds'].max():.2f}"
)

duplicate_mask = tracks.duplicated("sha256", keep=False)
duplicate_files = tracks.loc[duplicate_mask].sort_values(["sha256", "genre"])
conflicting_hashes = (
    tracks.groupby("sha256")["genre"].nunique().loc[lambda counts: counts > 1].index
)
print(f"\nFiles that are exact copies of another file: {int(duplicate_mask.sum())}")
print(f"Exact-content groups assigned conflicting genre labels: {len(conflicting_hashes)}")

if len(conflicting_hashes):
    print("Conflicting exact copies (excluded from this run):")
    display(duplicate_files[duplicate_files["sha256"].isin(conflicting_hashes)][
        ["track_name", "genre", "sha256"]
    ])

# A copy with conflicting labels has no unambiguous supervised target.
tracks = tracks.loc[~tracks["sha256"].isin(conflicting_hashes)].reset_index(drop=True)

print(f"\nFiles kept for modeling: {len(tracks)}")
display(tracks.groupby("genre").size().rename("usable_file_count").to_frame())

## 5. Prepare the full dataset for a shared split

This experiment uses the full eligible dataset for both models. It does not load either filter manifest. The smaller model's training subset is sampled only after the shared full-dataset train/validation/test split has been created.

In [ ]:
DATASET_LABELS = {
    "trained_on_full_dataset": "Full training data (control)",
    "trained_on_half_training_data": "50% random training subset",
}
full_dataset_tracks = tracks.reset_index(drop=True).copy()

if full_dataset_tracks["genre"].nunique() != len(CLASS_NAMES):
    raise ValueError("The full dataset does not contain all ten genres.")

print(f"Eligible full-dataset tracks: {len(full_dataset_tracks)}")
print("Eligible tracks per genre:")
display(
    full_dataset_tracks.groupby("genre")
    .size()
    .reindex(CLASS_NAMES)
    .rename("track_count")
    .to_frame()
)


## 6. Create one shared split, then sample half of the training data

The full dataset is split once at the original-track level. Whole SHA-256 groups stay together, and a per-genre subset-sum allocation targets 10% test and 9% validation; the rest is training. The second model then receives a seeded random subset targeting 50% of each genre's training tracks. The same validation and test tracks are used for both models. If duplicate groups make a target count impossible, the closest feasible count is used and reported.

In [ ]:
SPLIT_PERCENTAGES = {"train": 81, "validation": 9, "test": 10}
HALF_TRAINING_PERCENTAGE = 50


def choose_hash_groups_near_target(group_items, target_count, random_state):
    # Choose whole hash groups with a total track count nearest the target.
    shuffled_items = list(group_items)
    random.Random(random_state).shuffle(shuffled_items)

    # Dynamic programming over attainable track counts; each hash group is used once.
    attainable_counts = {0: ()}
    for group_hash, group_size in shuffled_items:
        previous_states = list(attainable_counts.items())
        for current_count, chosen_hashes in previous_states:
            next_count = current_count + int(group_size)
            if next_count not in attainable_counts:
                attainable_counts[next_count] = chosen_hashes + (group_hash,)

    best_count = min(
        attainable_counts,
        key=lambda count: (abs(count - target_count), count > target_count),
    )
    return set(attainable_counts[best_count]), best_count


def make_track_split(candidate_tracks, random_state=SEED):
    candidate_tracks = candidate_tracks.reset_index(drop=True).copy()
    class_counts = (
        candidate_tracks.groupby("genre").size().reindex(CLASS_NAMES, fill_value=0)
    )
    if (class_counts < 10).any():
        raise ValueError(
            "The full dataset needs at least 10 tracks in every genre for the "
            f"81/9/10 split. Counts: {class_counts.to_dict()}"
        )

    split_by_hash = {}
    allocation_rows = []
    test_fraction = SPLIT_PERCENTAGES["test"] / 100
    validation_fraction = SPLIT_PERCENTAGES["validation"] / 100

    for genre_index, genre in enumerate(CLASS_NAMES):
        genre_tracks = candidate_tracks.loc[candidate_tracks["genre"] == genre]
        group_sizes = genre_tracks.groupby("sha256").size()
        group_items = list(group_sizes.items())
        genre_total = len(genre_tracks)
        test_target = int(np.floor(genre_total * test_fraction + 0.5))
        validation_target = int(np.floor(genre_total * validation_fraction + 0.5))

        test_hashes, test_count = choose_hash_groups_near_target(
            group_items,
            test_target,
            random_state + genre_index * 100 + 1,
        )
        for group_hash in test_hashes:
            split_by_hash[group_hash] = "test"

        remaining_groups = [
            (group_hash, group_size)
            for group_hash, group_size in group_items
            if group_hash not in test_hashes
        ]
        validation_hashes, validation_count = choose_hash_groups_near_target(
            remaining_groups,
            validation_target,
            random_state + genre_index * 100 + 2,
        )
        for group_hash in validation_hashes:
            split_by_hash[group_hash] = "validation"
        for group_hash, _ in remaining_groups:
            if group_hash not in validation_hashes:
                split_by_hash[group_hash] = "train"

        train_count = genre_total - test_count - validation_count
        allocation_rows.append(
            {
                "genre": genre,
                "total_tracks": genre_total,
                "train_tracks": train_count,
                "train_percent": 100 * train_count / genre_total,
                "validation_tracks": validation_count,
                "validation_percent": 100 * validation_count / genre_total,
                "test_tracks": test_count,
                "test_percent": 100 * test_count / genre_total,
                "target_test_tracks": test_target,
                "target_validation_tracks": validation_target,
            }
        )

    if len(split_by_hash) != candidate_tracks["sha256"].nunique():
        raise AssertionError("A hash group did not receive exactly one split.")

    assignments = candidate_tracks[["genre", "sha256"]].copy()
    assignments["split"] = assignments["sha256"].map(split_by_hash)
    if assignments["split"].isna().any():
        raise AssertionError("Some tracks were not assigned to a split.")
    if assignments.groupby("sha256")["split"].nunique().max() != 1:
        raise AssertionError("An exact-content group crossed a split.")

    splits = {
        split_name: candidate_tracks.loc[
            assignments["split"] == split_name
        ].reset_index(drop=True)
        for split_name in ("train", "validation", "test")
    }
    return splits, assignments, pd.DataFrame(allocation_rows)


full_splits, full_split_assignments, genre_split_summary = make_track_split(
    full_dataset_tracks
)
train_tracks_full = full_splits["train"]
validation_tracks = full_splits["validation"]
test_tracks = full_splits["test"]

# Randomly select about half of the training tracks within each genre.
half_training_hashes = set()
half_training_rows = []
for genre_index, genre in enumerate(CLASS_NAMES):
    genre_train_tracks = train_tracks_full.loc[
        train_tracks_full["genre"] == genre
    ]
    group_sizes = genre_train_tracks.groupby("sha256").size()
    target_count = int(
        np.floor(len(genre_train_tracks) * HALF_TRAINING_PERCENTAGE / 100 + 0.5)
    )
    selected_hashes, selected_count = choose_hash_groups_near_target(
        list(group_sizes.items()),
        target_count,
        SEED + 10_000 + genre_index,
    )
    half_training_hashes.update(selected_hashes)
    half_training_rows.append(
        {
            "genre": genre,
            "full_train_tracks": len(genre_train_tracks),
            "half_target_tracks": target_count,
            "half_actual_tracks": selected_count,
            "selected_percent_of_full_train": (
                100 * selected_count / len(genre_train_tracks)
            ),
        }
    )

train_tracks_half = train_tracks_full.loc[
    train_tracks_full["sha256"].isin(half_training_hashes)
].reset_index(drop=True)

if len(train_tracks_half) >= len(train_tracks_full):
    raise AssertionError("The smaller training condition did not reduce the data.")
if not set(train_tracks_half["sha256"]).issubset(set(train_tracks_full["sha256"])):
    raise AssertionError("The smaller training data is not a subset of the full training data.")

print("Shared full-dataset split counts")
split_totals = pd.DataFrame(
    [
        {
            "split": split_name,
            "tracks": len(full_splits[split_name]),
            "percent_of_full_dataset": (
                100 * len(full_splits[split_name]) / len(full_dataset_tracks)
            ),
        }
        for split_name in ("train", "validation", "test")
    ]
).set_index("split")
display(split_totals.round(1))

print("Actual per-genre split counts and percentages")
display(genre_split_summary.set_index("genre").round(1))

print(
    f"Full training tracks: {len(train_tracks_full)}; "
    f"half-data training tracks: {len(train_tracks_half)} "
    f"({100 * len(train_tracks_half) / len(train_tracks_full):.1f}% of full training)"
)
print("Half-data selection by genre")
display(pd.DataFrame(half_training_rows).set_index("genre").round(1))


## 7. Convert tracks to cached log-mel spectrograms

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SAMPLES = SAMPLE_RATE * TRACK_SECONDS
EXPECTED_FRAMES = 1 + EXPECTED_SAMPLES // HOP_LENGTH
NUM_SEGMENTS = math.ceil(TRACK_SECONDS / SEGMENT_SECONDS)
FRAMES_PER_SEGMENT = math.ceil(EXPECTED_FRAMES / NUM_SEGMENTS)

FEATURE_CACHE_DIR = (
    ARTIFACTS_DIR
    / "gtzan_log_mel"
    / f"sr{SAMPLE_RATE}_mels{N_MELS}_fft{N_FFT}_hop{HOP_LENGTH}_sec{TRACK_SECONDS}_db{int(DB_RANGE)}"
)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def feature_path_for_hash(file_hash):
    return FEATURE_CACHE_DIR / file_hash[:2] / f"{file_hash}.npy"

def extract_log_mel(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=SAMPLE_RATE,
        mono=True,
        duration=TRACK_SECONDS,
    )
    if len(audio) < EXPECTED_SAMPLES:
        audio = np.pad(audio, (0, EXPECTED_SAMPLES - len(audio)))
    else:
        audio = audio[:EXPECTED_SAMPLES]

    power_mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    decibels = librosa.power_to_db(power_mel, ref=np.max, top_db=DB_RANGE)
    scaled = np.clip((decibels + DB_RANGE) / DB_RANGE, 0.0, 1.0).astype(np.float32)

    # Keep a predictable number of frames even if an audio backend differs by one frame.
    if scaled.shape[1] < EXPECTED_FRAMES:
        scaled = np.pad(scaled, ((0, 0), (0, EXPECTED_FRAMES - scaled.shape[1])))
    else:
        scaled = scaled[:, :EXPECTED_FRAMES]
    return scaled

unique_tracks = tracks.drop_duplicates("sha256")
for position, row in enumerate(unique_tracks.itertuples(index=False), start=1):
    cache_path = feature_path_for_hash(row.sha256)
    if not cache_path.exists():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, extract_log_mel(row.path))
    if position % 100 == 0 or position == len(unique_tracks):
        print(f"Prepared features for {position}/{len(unique_tracks)} unique tracks")

example_mel = np.load(feature_path_for_hash(unique_tracks.iloc[0]["sha256"]))
print(f"Feature cache: {FEATURE_CACHE_DIR.resolve()}")
print(f"One track feature shape: {example_mel.shape} (mel bands × time frames)")

## 8. Build loaders with shared validation and test data

Both models use the exact same validation and test track tables. Only their training tables differ: the control uses every track in the shared training split, and the smaller model uses the random half-size subset.

In [ ]:
class MelWindowDataset(Dataset):
    def __init__(self, track_table):
        self.track_rows = track_table.to_dict(orient="records")
        self.examples = [
            (track_index, segment_index)
            for track_index, row in enumerate(self.track_rows)
            for segment_index in range(NUM_SEGMENTS)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, item_index):
        track_index, segment_index = self.examples[item_index]
        row = self.track_rows[track_index]
        mel = np.load(feature_path_for_hash(row["sha256"]), mmap_mode="r")

        start = segment_index * FRAMES_PER_SEGMENT
        window = np.array(
            mel[:, start : start + FRAMES_PER_SEGMENT],
            dtype=np.float32,
            copy=True,
        )
        if window.shape[1] < FRAMES_PER_SEGMENT:
            window = np.pad(
                window,
                ((0, 0), (0, FRAMES_PER_SEGMENT - window.shape[1])),
            )

        features = torch.from_numpy(window[None, :, :])
        label = torch.tensor(row["label_id"], dtype=torch.long)
        source_track = torch.tensor(track_index, dtype=torch.long)
        return features, label, source_track


def make_train_loader(dataset):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )


full_train_dataset = MelWindowDataset(train_tracks_full)
half_train_dataset = MelWindowDataset(train_tracks_half)
shared_validation_dataset = MelWindowDataset(validation_tracks)
shared_test_dataset = MelWindowDataset(test_tracks)

shared_validation_loader = DataLoader(
    shared_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
shared_test_loader = DataLoader(
    shared_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

EXPERIMENT_RUNS = {
    "trained_on_full_dataset": {
        "train_dataset": full_train_dataset,
        "validation_dataset": shared_validation_dataset,
        "test_dataset": shared_test_dataset,
        "train_loader": make_train_loader(full_train_dataset),
        "validation_loader": shared_validation_loader,
        "test_loader": shared_test_loader,
    },
    "trained_on_half_training_data": {
        "train_dataset": half_train_dataset,
        "validation_dataset": shared_validation_dataset,
        "test_dataset": shared_test_dataset,
        "train_loader": make_train_loader(half_train_dataset),
        "validation_loader": shared_validation_loader,
        "test_loader": shared_test_loader,
    },
}

print(f"Full-model training windows: {len(full_train_dataset)}")
print(f"Half-data training windows: {len(half_train_dataset)}")
print(f"Shared validation windows: {len(shared_validation_dataset)}")
print(f"Shared test windows: {len(shared_test_dataset)}")


## 9. Define the Phase 2.2 CNN

In [ ]:
class SmallAudioCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.filters = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.filters(inputs))

## 10. Aggregate window probabilities to track predictions

In [ ]:
@torch.inference_mode()
def collect_predictions(model, data_loader, dataset):
    model.eval()
    all_probabilities = []
    all_window_labels = []
    all_track_indices = []

    for features, labels, track_indices in data_loader:
        logits = model(features.to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probabilities)
        all_window_labels.append(labels.numpy())
        all_track_indices.append(track_indices.numpy())

    window_probabilities = np.concatenate(all_probabilities)
    window_true = np.concatenate(all_window_labels)
    window_track_indices = np.concatenate(all_track_indices)

    track_probabilities = np.zeros(
        (len(dataset.track_rows), len(CLASS_NAMES)),
        dtype=np.float64,
    )
    window_counts = np.zeros(len(dataset.track_rows), dtype=np.int64)
    np.add.at(track_probabilities, window_track_indices, window_probabilities)
    np.add.at(window_counts, window_track_indices, 1)
    track_probabilities /= window_counts[:, None]

    track_true = np.asarray(
        [row["label_id"] for row in dataset.track_rows],
        dtype=np.int64,
    )
    track_predicted = track_probabilities.argmax(axis=1)

    return {
        "window_probabilities": window_probabilities,
        "window_true": window_true,
        "window_predicted": window_probabilities.argmax(axis=1),
        "window_track_indices": window_track_indices,
        "track_probabilities": track_probabilities,
        "track_true": track_true,
        "track_predicted": track_predicted,
    }

## 11. Train both models with the same protocol

The model architecture, initialization seed, loss, optimizer, learning rate, scheduler, epoch limit, patience, and validation checkpoint rule match the Phase 2.2 workflow. Both runs select checkpoints on the same validation tracks and are evaluated on the same test tracks. Checkpoints are saved separately under artifacts/phase_2_2_experiment_6.

In [ ]:
def reset_random_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_model(run_name, train_loader, validation_loader, validation_dataset):
    """Train the reference CNN and select its checkpoint on this run’s validation tracks."""
    reset_random_seeds(SEED)
    model = SmallAudioCNN(number_of_classes=len(CLASS_NAMES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    history = []
    best_score = (-1.0, -1.0)
    best_epoch = 0
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct_windows = 0
        seen_windows = 0

        for features, labels_batch, _ in train_loader:
            features = features.to(DEVICE)
            labels_batch = labels_batch.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels_batch)
            loss.backward()
            optimizer.step()

            batch_size = labels_batch.size(0)
            total_loss += loss.item() * batch_size
            correct_windows += (logits.argmax(dim=1) == labels_batch).sum().item()
            seen_windows += batch_size

        validation_result = collect_predictions(
            model, validation_loader, validation_dataset
        )
        validation_accuracy = accuracy_score(
            validation_result["track_true"], validation_result["track_predicted"]
        )
        validation_macro_f1 = f1_score(
            validation_result["track_true"],
            validation_result["track_predicted"],
            average="macro",
            zero_division=0,
        )
        scheduler.step(validation_accuracy)

        epoch_record = {
            "epoch": epoch,
            "train_loss": total_loss / seen_windows,
            "train_window_accuracy": correct_windows / seen_windows,
            "validation_track_accuracy": validation_accuracy,
            "validation_track_macro_f1": validation_macro_f1,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        score = (validation_accuracy, validation_macro_f1)
        if score > best_score:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
        else:
            epochs_without_improvement += 1

        print(
            f"{run_name} | Epoch {epoch:02d}/{EPOCHS} | "
            f"loss {epoch_record['train_loss']:.4f} | "
            f"train window acc {epoch_record['train_window_accuracy']:.3f} | "
            f"validation track acc {validation_accuracy:.3f} | "
            f"validation macro-F1 {validation_macro_f1:.3f} | "
            f"lr {epoch_record['learning_rate']:.2e}"
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"{run_name}: early stopping after {PATIENCE} epochs without improvement.")
            break

    if best_state is None:
        raise RuntimeError(f"{run_name} training did not produce a checkpoint.")

    model.load_state_dict(best_state)
    model.eval()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CHECKPOINT_DIR / f"{run_name.lower().replace(' ', '_')}.pt"
    torch.save(
        {
            "model_state_dict": best_state,
            "class_names": CLASS_NAMES,
            "best_epoch": best_epoch,
            "sample_rate": SAMPLE_RATE,
            "track_seconds": TRACK_SECONDS,
            "segment_seconds": SEGMENT_SECONDS,
            "n_mels": N_MELS,
            "n_fft": N_FFT,
            "hop_length": HOP_LENGTH,
            "seed": SEED,
        },
        checkpoint_path,
    )
    print(f"{run_name}: best validation checkpoint at epoch {best_epoch}; saved to {checkpoint_path}")
    return model, pd.DataFrame(history), best_epoch


In [ ]:
experiment_models = {}
experiment_histories = {}
experiment_best_epochs = {}

for run_name, run_data in EXPERIMENT_RUNS.items():
    print(f"\nStarting {DATASET_LABELS[run_name]} run ({run_name})")
    model, history, best_epoch = train_model(
        run_name,
        run_data["train_loader"],
        run_data["validation_loader"],
        run_data["validation_dataset"],
    )
    experiment_models[run_name] = model
    experiment_histories[run_name] = history
    experiment_best_epochs[run_name] = best_epoch
    print(f"{DATASET_LABELS[run_name]} best epoch: {best_epoch}\n")


## 12. Compare learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run_name, history_table in experiment_histories.items():
    label = DATASET_LABELS[run_name]
    axes[0].plot(
        history_table["epoch"],
        history_table["train_loss"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        history_table["epoch"],
        history_table["validation_track_accuracy"],
        marker="o",
        label=f"{label} accuracy",
    )
    axes[1].plot(
        history_table["epoch"],
        history_table["validation_track_macro_f1"],
        marker="s",
        linestyle="--",
        label=f"{label} macro-F1",
    )

axes[0].set_title("Training loss (5-second windows)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_title("Validation track metrics (shared validation tracks)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


## 13. Compare held-out test metrics, per-genre scores, and confusion matrices

Primary metrics use one prediction per full track after averaging the six window probabilities. Both models are evaluated on the same held-out tracks, so test supports and per-track correctness outcomes are directly comparable. Window-level accuracy and macro-F1 are secondary diagnostics, matching the baseline.

In [ ]:
def summarize_metrics(result):
    return {
        "test_tracks": len(result["track_true"]),
        "accuracy": accuracy_score(result["track_true"], result["track_predicted"]),
        "balanced_accuracy": balanced_accuracy_score(
            result["track_true"], result["track_predicted"]
        ),
        "macro_f1": f1_score(
            result["track_true"],
            result["track_predicted"],
            average="macro",
            zero_division=0,
        ),
        "window_accuracy": accuracy_score(
            result["window_true"], result["window_predicted"]
        ),
        "window_macro_f1": f1_score(
            result["window_true"],
            result["window_predicted"],
            average="macro",
            zero_division=0,
        ),
    }


experiment_test_results = {}
experiment_reports = {}
metric_rows = {}
for run_name, run_data in EXPERIMENT_RUNS.items():
    result = collect_predictions(
        experiment_models[run_name],
        run_data["test_loader"],
        run_data["test_dataset"],
    )
    experiment_test_results[run_name] = result
    metric_rows[run_name] = summarize_metrics(result)
    experiment_reports[run_name] = classification_report(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

metric_comparison = pd.DataFrame(metric_rows).T
metric_comparison.index = [
    DATASET_LABELS[run_name] for run_name in metric_comparison.index
]
control_label = DATASET_LABELS["trained_on_full_dataset"]
for metric_name in ("accuracy", "balanced_accuracy", "macro_f1"):
    metric_comparison[f"{metric_name}_delta_vs_control"] = (
        metric_comparison[metric_name] - metric_comparison.loc[control_label, metric_name]
    )

print("Held-out TEST metrics (identical test tracks for both models)")
display(metric_comparison.round(3))


def make_class_metric_comparison(metric_name):
    rows = []
    for genre in CLASS_NAMES:
        full_score = experiment_reports["trained_on_full_dataset"][genre][metric_name]
        half_score = experiment_reports["trained_on_half_training_data"][genre][metric_name]
        rows.append(
            {
                "genre": genre,
                "full_training_data": full_score,
                "half_training_data": half_score,
                "half_minus_full": half_score - full_score,
                "test_support": experiment_reports["trained_on_full_dataset"][genre]["support"],
            }
        )
    return pd.DataFrame(rows).set_index("genre")

for metric_name in ("precision", "recall", "f1-score"):
    print(f"Per-genre {metric_name} on the shared test set")
    display(make_class_metric_comparison(metric_name).round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, run_name in zip(axes, EXPERIMENT_RUNS):
    result = experiment_test_results[run_name]
    ConfusionMatrixDisplay.from_predictions(
        result["track_true"],
        result["track_predicted"],
        labels=np.arange(len(CLASS_NAMES)),
        display_labels=CLASS_NAMES,
        cmap="Blues",
        xticks_rotation=45,
        values_format="d",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(DATASET_LABELS[run_name])
plt.tight_layout()
plt.show()


## 14. Compare paired test outcomes and interpret the result

If the half-data model scores better on this shared test set, that supports the idea that it can learn as well or better from fewer examples under this seed and split. It does not by itself prove that a large dataset overwhelms the model; a different random half or seed may change the result. If the half-data model scores worse, the result supports the value of the additional training examples for this setup.

The validation curves show whether the smaller model reaches a strong score sooner or plateaus differently. Both models use the same validation set for checkpoint selection. This experiment does not test staged training on a second dataset; that would be a separate follow-up using the smaller model's checkpoint and then continuing training.

In [ ]:
full_name = "trained_on_full_dataset"
half_name = "trained_on_half_training_data"
full_label = DATASET_LABELS[full_name]
half_label = DATASET_LABELS[half_name]
full_result = experiment_test_results[full_name]
half_result = experiment_test_results[half_name]

if not np.array_equal(full_result["track_true"], half_result["track_true"]):
    raise AssertionError("The two runs do not share the same test labels.")
full_test_names = [
    row["track_name"] for row in EXPERIMENT_RUNS[full_name]["test_dataset"].track_rows
]
half_test_names = [
    row["track_name"] for row in EXPERIMENT_RUNS[half_name]["test_dataset"].track_rows
]
if full_test_names != half_test_names:
    raise AssertionError("The two runs do not share the same ordered test tracks.")

full_correct = full_result["track_predicted"] == full_result["track_true"]
half_correct = half_result["track_predicted"] == half_result["track_true"]
paired_outcomes = pd.Series(
    np.select(
        [
            full_correct & half_correct,
            full_correct & ~half_correct,
            ~full_correct & half_correct,
        ],
        ["both correct", "full-data only correct", "half-data only correct"],
        default="both wrong",
    ),
    name="paired_outcome",
)
print("Paired correctness outcomes on the shared test tracks")
display(paired_outcomes.value_counts().rename("track_count").to_frame())

paired_predictions = pd.DataFrame(
    {
        "track": [row["track_name"] for row in EXPERIMENT_RUNS[full_name]["test_dataset"].track_rows],
        "true_genre": [CLASS_NAMES[int(value)] for value in full_result["track_true"]],
        "full_prediction": [CLASS_NAMES[int(value)] for value in full_result["track_predicted"]],
        "half_data_prediction": [CLASS_NAMES[int(value)] for value in half_result["track_predicted"]],
        "full_correct": full_correct,
        "half_data_correct": half_correct,
        "paired_outcome": paired_outcomes,
    }
)
print("Tracks where the models differ in correctness")
display(
    paired_predictions.loc[
        paired_predictions["full_correct"] != paired_predictions["half_data_correct"]
    ].reset_index(drop=True)
)

accuracy_change = metric_comparison.loc[half_label, "accuracy_delta_vs_control"]
print("\nInterpretation using held-out track accuracy")
if accuracy_change > 0:
    print(
        f"The half-data model scored higher by {accuracy_change:+.3f} on the shared "
        "test set. This supports the smaller-data hypothesis for this split and seed."
    )
elif accuracy_change < 0:
    print(
        f"The half-data model scored lower by {accuracy_change:+.3f} on the shared "
        "test set. The additional training examples helped under this split and seed."
    )
else:
    print("The models tied in accuracy on the shared test set.")

print(
    "Review balanced accuracy, macro-F1, per-genre precision/recall, the validation "
    "curves, and the paired outcomes before drawing a broader conclusion."
)
